# LAB-HW-08 — Small FlyBrain Replay

**One new thing today:** run a small event-driven FlyBrain teaching network in real programmable logic (PL), then compare the complete hardware trace against the same deterministic Python oracle.

Prerequisites: LSN-012, LAB-HW-06, LAB-HW-07.

**Project Trace:** RMD-013 · T-HW-008/T-HW-011

This is the first Lab where several concepts you already learned work together on the board. It introduces no DDR and no performance claim.

## 1. Keep the semantic boundary straight

LAB-HW-08 replays the exact **Lesson-12 teaching event machine**.

Lesson 12 explicitly says that machine is **not** the formal LIF neuron model. It omits leak, refractory behavior, final fixed-point numerics, concurrent target-write conflicts, and formal valid/ready timing.

Therefore a PASS here means:

> “the already verified Lesson-12 event-causality network produces the same deterministic trace in KV260 PL.”

It does **not** mean formal MOD-004~009 are complete.

## 2. Freeze the exact network before touching RTL

Source of truth:

`boards/kv260/fixtures/lab08_four_neuron_replay_v1.json`

The fixture is the same four-neuron Lesson-12 network:

- 0 → 1, weight +2
- 0 → 2, weight +1
- 1 → 3, weight +2
- 2 → 3, weight +1
- thresholds: `[99, 2, 1, 3]`
- initial accumulator: `[0,0,0,0]`
- initial input queue: `[0]`

Expected spike order: `[0,1,2,3]`.

Expected final accumulator state: `[0,0,0,0]`.

The fixture is deterministic and contains no PRNG. We do not invent a meaningless seed.

## 3. Let Python define the expected trace

Run:

```bash
python boards/kv260/runtime/lab08_replay_reference.py \
  --fixture boards/kv260/fixtures/lab08_four_neuron_replay_v1.json \
  --json-out /tmp/lab-hw-08-reference.json
```

It must end in `STATUS=PASS`.

The four weighted-event words are frozen as:

```console
0x01020201  # 0 -> 1, +2, accumulator reaches 2, spike
0x02010101  # 0 -> 2, +1, accumulator reaches 1, spike
0x13020200  # 1 -> 3, +2, accumulator reaches 2, no spike
0x23010301  # 2 -> 3, +1, accumulator reaches 3, spike
```

RTL must match this oracle; the RTL is not allowed to redefine it.

## 4. What runs where?

<svg xmlns="http://www.w3.org/2000/svg" width="1000" height="330" viewBox="0 0 1000 330" role="img" aria-label="LAB-HW-08 host control and replay paths">
  <rect x="25" y="110" width="150" height="90" rx="10" fill="#eef" stroke="#333"/>
  <text x="100" y="143" text-anchor="middle" font-size="14">PS / Linux</text>
  <text x="100" y="168" text-anchor="middle" font-size="12">Python checker</text>
  <rect x="220" y="40" width="180" height="85" rx="10" fill="#efe" stroke="#333"/>
  <text x="310" y="72" text-anchor="middle" font-size="14">AXI BRAM Controller</text>
  <text x="310" y="98" text-anchor="middle" font-size="12">0xA0000000</text>
  <rect x="220" y="210" width="180" height="85" rx="10" fill="#efe" stroke="#333"/>
  <text x="310" y="242" text-anchor="middle" font-size="14">AXI GPIO</text>
  <text x="310" y="268" text-anchor="middle" font-size="12">0xA0010000</text>
  <rect x="455" y="40" width="210" height="85" rx="10" fill="#fee" stroke="#333"/>
  <text x="560" y="72" text-anchor="middle" font-size="14">shared state / trace BRAM</text>
  <text x="560" y="98" text-anchor="middle" font-size="12">4 KiB</text>
  <rect x="455" y="210" width="210" height="85" rx="10" fill="#fee" stroke="#333"/>
  <text x="560" y="242" text-anchor="middle" font-size="14">small replay engine</text>
  <text x="560" y="268" text-anchor="middle" font-size="12">queue → lookup → update</text>
  <rect x="720" y="110" width="240" height="90" rx="10" fill="#fff8dc" stroke="#333"/>
  <text x="840" y="140" text-anchor="middle" font-size="14">spike + weighted-event trace</text>
  <text x="840" y="165" text-anchor="middle" font-size="12">written into reserved BRAM words</text>
  <path d="M175 140 L220 85 M175 175 L220 250 M400 82 L455 82 M400 252 L455 252 M665 250 L720 175 M665 82 L720 140" stroke="#333" stroke-width="2" fill="none"/>
</svg>

Important rule: **while `busy=1`, the host does not access BRAM**. This avoids introducing concurrent BRAM arbitration in this Lab.

## 5. Read the trace as memory

The 4 KiB BRAM window keeps the LAB-HW-07 base `0xA0000000`.

Reserved word indices:

| words | meaning |
|---|---|
| 0..3 | final accumulator state |
| 16..19 | spike order |
| 20 | spike count |
| 32..35 | encoded weighted-event trace |
| 36 | event count |

Control/status stays at the LAB-HW-06 GPIO base `0xA0010000`.

Status bits:

- bit 0: `busy`
- bit 1: `done`
- bit 2: `error`
- bits 7:4: spike count
- bits 15:8: event count

## 6. Prove RTL replay before board build

```bash
iverilog -g2012 \
  -s kv260_small_replay_engine_tb \
  -o /tmp/lab08_replay \
  boards/kv260/rtl/kv260_replay_state_store.sv \
  boards/kv260/rtl/kv260_small_replay_engine.sv \
  boards/kv260/tb/kv260_small_replay_engine_tb.sv

vvp /tmp/lab08_replay
```

Expected final line:

`PASS: LAB-HW-08 Lesson-12 four-neuron replay trace`

This simulation checks final state, spike order, counts, and all four encoded weighted events.

## 7. Exercise the differential checker without hardware

```bash
python boards/kv260/runtime/small_replay_mmio.py \
  --fixture boards/kv260/fixtures/lab08_four_neuron_replay_v1.json \
  --dry-run \
  --json-out /tmp/lab-hw-08-dry-run.json
```

It must print both `DIFFERENTIAL=PASS` and `STATUS=PASS`.

Dry-run proves checker/oracle behavior. It does **not** prove a KV260 or Vivado build.

## 8. Build and program LAB-HW-08

Development host:

```bash
vivado -mode batch -nojournal \
  -log lab-hw-08-build.log \
  -source boards/kv260/scripts/build_lab08_small_replay.tcl
```

The build blocks bitstream generation on:

- DRC errors;
- missing setup/hold timing paths;
- negative setup/hold slack;
- zero RAMB18/RAMB36 primitives.

Expected bitstream:

`build/kv260/lab-hw-08/kv260_small_replay.bit`

Keep Linux running, unload an active Kria app if required, then program with the shared `program_bitstream.tcl` exactly as in LAB-HW-06/07.

## 9. Run the real differential replay

Copy these three files to the runtime host by an already-working method:

- `small_replay_mmio.py`
- `lab08_replay_reference.py`
- `lab08_four_neuron_replay_v1.json`

Then:

```bash
sudo python3 /tmp/small_replay_mmio.py \
  --fixture /tmp/lab08_four_neuron_replay_v1.json \
  --json-out /tmp/lab-hw-08-trace.json
```

The checker:

1. computes the Python expected trace;
2. confirms the PL engine is idle;
3. initializes state and clears stale trace words;
4. raises `start`;
5. polls status only while `busy=1`;
6. after completion, reads the BRAM trace;
7. compares every field.

Physical PASS requires `DIFFERENTIAL=PASS` and `STATUS=PASS`.

## 10. Failure classes

Host/replay failures are separated:

- `FIXTURE_ORACLE_INVALID`
- `TRANSPORT_DEVICE_MISSING`
- `TRANSPORT_REQUIRES_ROOT`
- `TRANSPORT_PERMISSION_OR_POLICY`
- `TRANSPORT_MMAP_FAILED`
- `ENGINE_ALREADY_BUSY`
- `ENGINE_REPORTED_ERROR`
- `ENGINE_TIMEOUT`
- `REPLAY_DIFFERENTIAL_MISMATCH`

A transport failure is not an algorithm mismatch. An algorithm mismatch is not a timing/resource failure.

As before, if Ubuntu policy blocks `/dev/mem`, do not weaken system security merely to force PASS.

## 11. Expected Evidence / Save Evidence

Retain:

- fixture SHA-256;
- `lab08_replay_reference.py` SHA-256;
- `small_replay_mmio.py` SHA-256;
- reference JSON;
- Icarus PASS output;
- Vivado build/timing/utilization/DRC reports;
- bitstream SHA-256;
- programming log;
- complete physical runtime stdout;
- `lab-hw-08-trace.json`;
- exact expected and observed spike/state/event traces;
- Git commit, Ubuntu/kernel identity, board/carrier revision, date.

A cloud-CI PASS is useful engineering evidence, but it is not a physical T-HW-008 PASS.

## 12. Debug in layers

When the replay fails, check in this order:

1. fixture hash and Python oracle;
2. open-source RTL simulation;
3. Vivado DRC/timing/resource reports;
4. bitstream/programming identity;
5. Linux transport/status;
6. only then the PL differential trace.

This order prevents a stale fixture or transport failure from being misdiagnosed as “the neural network is wrong.”

## 13. Human Check

Explain without looking at code:

1. Why is the JSON fixture the network source of truth?
2. Why is the Python replay run before RTL?
3. Why is `[0,1,2,3]` valid only for this Lesson-12 teaching machine?
4. Why does the host avoid BRAM while `busy=1`?
5. Why compare weighted-event trace instead of only final state?
6. What is the difference between `ENGINE_TIMEOUT` and `REPLAY_DIFFERENTIAL_MISMATCH`?
7. Why does HW-08 still not mean formal MOD-004~009 are complete?

## 14. Engineering handoff

After LAB-HW-08, the third Physical-Lab stage is complete in teaching/CI scope: HW-06 established PS↔PL control, HW-07 established real on-chip neuron-state memory, and HW-08 adds deterministic small-event-network replay in PL.

The next stage changes the problem: LAB-HW-09 moves known data through external DDR and proves integrity before measuring anything.

## 15. Official basis

- Repository teaching source: `lessons/en/12_one_spike_journey.ipynb`, especially cells `e1205b`, `e1206`, `e1207`, and `e1208`.
- Engineering authority: `docs/en/RMD.md`, `docs/en/MDD.md`, and `docs/en/TDD.md`.
- KV260 address-path basis: the already frozen LAB-HW-06/07 PS `M_AXI_HPM0_FPD`, SmartConnect, AXI GPIO, and AXI BRAM Controller teaching paths.

This Lab preserves the Lesson-12 semantics instead of silently replacing them with a formal LIF model.